In [ ]:
# Import necessary libraries
import pandas as pd
from collections import defaultdict
import geopandas as gpd
from shapely import wkt
import shapely

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pickle
%matplotlib inline
import datetime
import time
import numpy as np
import xml.etree.ElementTree as ET 
import matsim

import utm
from shapely.geometry import Polygon, Point
import gzip
from matplotlib import cm

import matplotlib.ticker as ticker
import matplotlib.font_manager as font_manager
import matplotlib as mpl
from matplotlib.lines import Line2D
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap
import json



In [ ]:
# =========================================================================================
# NOTEBOOK NOTE:
# -----------------------------------------------------------------------------------------
# This file is a quick and dirty workbench to get results out of the batch runs.
# It is messy on purpose. Stability and correctness first, cleanup later.
# Sorry to anyone reading this :DD Quick and dirty for evaluation. No Time, Paper Deadline...
# It’s not clean, not modular, and definitely not pretty.
# The only goal right now: make the plots look right and get the paper submitted.
# =============================================================================
# NOTE
# -----------------------------------------------------------------------------
#
# What this does
# - Discovers MATSim runs in BATCH_DIR
# - Parses events and carriers
# - Builds per vehicle stats and EV assignments
# - Computes emissions (drive, idle, cold) and aggregates to 15 min bins
# - Writes a bunch of .pkl bundles next to the runs
#
# Known mess
# - Hard coded paths
# - Globals all over the place
# - Mixed responsibilities inside run_batch
# - Missing guards for some None cases
#
# TODO after the deadline
# [ ] Move helpers into utils modules
# [ ] Replace prints with structured logging
# [ ] Add type hints and docstrings
# [ ] Centralize config and paths
# [ ] Unit tests for parsing and clipping
# [ ] Package this into a clean CLI
# =============================================================================

In [ ]:
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)

def determine_area_type(raumtyp):
    if raumtyp in [1, 2, 3]:
        return 'Urban'
    elif raumtyp in [4, 5, 6]:
        return 'Suburban'
    elif raumtyp in [7, 8]:
        return 'Rural'
    else:
        return 'unknown'
    
category_dict = {
    "1": "Metropolitan Center",
    "2": "High-Density Residential Use",
    "3": "Dense Mixed Use",
    "4": "Residential Use",
    "5": "Industrial Use",
    "6": "Urbanized Periphery",
    "7": "Rural with Industrial Influence",
    "8": "Rural without Industrial Influence"
}


# Wende die Funktion auf die Spalte raumtyp an und erstelle die neue Spalte area_type
regionclusters['area_type_agg'] = regionclusters['raumtyp'].apply(determine_area_type)
regionclusters['area_type'] = regionclusters['raumtyp'].astype(str).map(category_dict)

# Explode the multipolygons into individual polygons
regionclusters_split = regionclusters.explode(index_parts=False)

# Reset index to clean up the DataFrame
regionclusters_split.reset_index(drop=True, inplace=True)

# Ensure regionclusters_split has a unique 'id' column for each geometry
if 'id' not in regionclusters_split.columns:
    regionclusters_split = regionclusters_split.reset_index().rename(columns={'index': 'id'})


# Read the CSV file
folder = "input/"
areas = pd.read_csv( folder + "plz_areas.csv")  

# Convert the 'WKT' column to Shapely MultiPolygon geometry
areas['geometry'] = areas['WKT'].apply(lambda wkt_str: wkt.loads(wkt_str))

# Create a GeoDataFrame
gdf_areas = gpd.GeoDataFrame(areas, geometry='geometry')

# Set the coordinate reference system (CRS)
gdf_areas.crs = 'EPSG:25832'  # Set the appropriate CRS if it's different


# Import Network
# MATSim network als Geopandas dataframe inkl. LINESTRINGS einlesen
# network = matsim.read_network('D:\\Hannover Daten\\MatSim\\Network XT\\car_network_encfix.xml.gz').as_geo().set_crs('epsg:25832') # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
network = matsim.read_network('input/car_network_encfix.xml.gz').as_geo().set_crs('epsg:25832')
# dict mit Netzwerk Link Längen
# link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
network.head()

# falls vorhanden Ergebnisse der nächsten Zellen laden, sonst überspringen
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)
with open('input/networkplus.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    networkplus = pd.read_pickle(pickle_file)     


link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
link_geometry = network[['link_id', 'geometry']].set_index('link_id').to_dict()['geometry']
# dict mit Netzwerk Link Raumtypen
link_raumtyp = networkplus['raumtyp'].to_dict()

# Create a dictionary linking each area type code to its description
area_type_dict = {
    1: "Metropolitan Center",
    2: "High-Density Residential Use",
    3: "Dense Mixed Use",
    4: "Residential Use",
    5: "Industrial Use",
    6: "Urbanized Periphery",
    7: "Rural with Industrial Influence",
    8: "Rural without Industrial Influence"
}

from matplotlib.colors import to_hex
# Define the desired order of legend labels in English
# Create a dictionary linking each category to a color



category_color_dict = {
    "Metropolitan Center": "#482878",
    "High-Density Residential Use": "#3f4989",
    "Dense Mixed Use": "#31688e",
    "Residential Use": "#26828e",
    "Industrial Use": "#1f9e89",
    "Urbanized Periphery": "#35b779",
    "Rural with Industrial Influence": "#6fce58",
    "Rural without Industrial Influence": "#b5de2b"
}



category_color_dict_num = {
    0: "#482878",
    1: "#3f4989",
    2: "#31688e",
    3: "#26828e",
    4: "#1f9e89",
    5: "#35b779",
    6: "#6fce58",
    7: "#b5de2b"
}



# Create a dictionary linking each category to a color
category_color = {category: i / (len(category_color_dict) - 1) for i, category in enumerate(category_color_dict)}

# Create the colormap
cmap = LinearSegmentedColormap.from_list("custom_cmap", sns.color_palette("viridis_r", len(category_color_dict))[::-1])

# Convert the colormap to a dictionary
palette_dict = {cat: to_hex(cmap(category_color[cat])) for cat in category_color_dict}


In [54]:
# def Methoden

# vehicle string pattern
def parse_events(event_file):
    """
    This function parses events from a given event file. It filters out events of type 'left link' and 'actstart'.
    It also counts the number of events for different types of vehicles and stores the last link for each vehicle.
    
    Args:
        event_file (str): The path to the event file to be parsed.
        
    Returns:
        vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
        service_events (dict): A dictionary mapping each person to a list of service events.
        network_volumes (DataFrame): A DataFrame containing the counts of each type of vehicle on each link.
    """
    
    # Only returns events of type 'left link' and 'actstart:
    events = matsim.event_reader(event_file, types='left link,actstart')

    # defaultdict creates a blank dict entry on first reference; similar to {} but more friendly
    link_counts_trucks = defaultdict(int)
    link_counts_vans = defaultdict(int)
    link_counts_bikes = defaultdict(int)
    link_counts_cars = defaultdict(int)

    vehicle_tour = defaultdict(list)
    service_events = defaultdict(list)
    last_link = defaultdict(str)
    
    vehicle_services = defaultdict(list)

    for event in events:
        # skip next day events
        if event['time'] > DAYEND: continue

        # on link left
        if event['type'] == 'left link':
            vehicle = event['vehicle']
            # fix broken ID encodings in basecase
            # vehicle = vehicle.encode('Windows 1252').decode('UTF-8') # für die Auswertung egal
            # identify CEP truck
            if '_Supply_Vehicle_' in vehicle or '_veh_supply_' in vehicle:
                if event['link'] != last_link[vehicle]:
                    link_counts_trucks[event['link']] += 1
                    vehicle_tour[vehicle].append(event['link'])
                    last_link[vehicle] = event['link']
            # identify vans (CEP & eGrocery)
            elif '_CEP_Vehicle_' in vehicle or '_veh_cep_' in vehicle or '_egrocery_van_' in vehicle:
                if event['link'] != last_link[vehicle]:
                    link_counts_vans[event['link']] += 1
                    vehicle_tour[vehicle].append(event['link'])
                    last_link[vehicle] = event['link']
            # identify bikes
            elif '_cargoBike_' in vehicle or '_cargobike_' in vehicle:
                if event['link'] != last_link[vehicle]:
                    link_counts_bikes[event['link']] += 1
                    vehicle_tour[vehicle].append(event['link'])
                    last_link[vehicle] = event['link']
        elif event['type'] == 'actstart':
            person = event['person']
            # fix broken ID encodings in basecase
            # person = person.encode('Windows 1252').decode('UTF-8') # für die Auswertung egal
            # catch last tour link
            if event['actType'] == 'end':
            # identify CEP truck persons
                if '_Supply_Vehicle_' in person or '_veh_supply_' in person:
                    link_counts_trucks[event['link']] += 1
                    vehicle_tour[person].append(event['link'])
                # identify van persons
                elif '_CEP_Vehicle_' in person or '_veh_cep_' in person or '_egrocery_van_' in person:
                    link_counts_vans[event['link']] += 1
                    vehicle_tour[person].append(event['link'])
                # identify bike persons
                elif '_cargoBike_' in person or '_cargobike_' in vehicle:
                    link_counts_bikes[event['link']] += 1
                    vehicle_tour[person].append(event['link'])
            # catch services
            elif event['actType'] == 'service':
                service_events[person].append(event['link'])
                vehicle_services[person].append(event)
    


    # convert link_counts dict to a pandas dataframe
    link_counts_trucks = pd.DataFrame.from_dict(link_counts_trucks, orient='index', columns=['truck_count']).rename_axis('link_id')
    link_counts_vans = pd.DataFrame.from_dict(link_counts_vans, orient='index', columns=['van_count']).rename_axis('link_id')
    link_counts_bikes = pd.DataFrame.from_dict(link_counts_bikes, orient='index', columns=['bike_count']).rename_axis('link_id')
    link_counts_cars = pd.DataFrame.from_dict(link_counts_cars, orient='index', columns=['car_count']).rename_axis('link_id')

    # attach counts to our Geopandas network from above
    network_volumes = network.merge(link_counts_trucks, on='link_id', how='left').merge(link_counts_vans, on='link_id', how='left').merge(link_counts_bikes, on='link_id', how='left').merge(link_counts_cars, on='link_id', how='left')
    # network_volumes['car_count'] = network_volumes['car_count'].fillna(0)
    # network_volumes['truck_count'] = network_volumes['truck_count'].fillna(0)
    # network_volumes['van_count'] = network_volumes['van_count'].fillna(0)
    # network_volumes['bike_count'] = network_volumes['bike_count'].fillna(0)
    numeric_cols = ['car_count', 'truck_count', 'van_count', 'bike_count']
    network_volumes[numeric_cols] = network_volumes[numeric_cols].apply(pd.to_numeric, errors='coerce')
    network_volumes[numeric_cols] = network_volumes[numeric_cols].fillna(0)
    network_volumes['total_count'] = network_volumes['truck_count'] + network_volumes['van_count'] + network_volumes['bike_count']

    return vehicle_tour, service_events, network_volumes, vehicle_services


def vehicle_stats(vehicle_tour, service_events):
    """
    This function calculates statistics for each vehicle, including the number of services, total tour length,
    distance to the first service, and distribution of services over different types of areas.
    
    Args:
        vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
        service_events (dict): A dictionary mapping each person to a list of service events.
        
    Returns:
        veh_df (DataFrame): A DataFrame containing statistics for each vehicle.
    """
        
    # init Ergebnis Objekte
    vehicle_stats = list()

    # Loop über Fahrzeuge und ihre Services
    for veh, ser in service_events.items():
        # Unterscheidung Fahrzeugtypen
        if '_Supply_Vehicle_' in veh or '_veh_supply_' in veh:
            c = 'truck'            
        if '_CEP_Vehicle_' in veh or '_veh_cep_' in veh or '_egrocery_van_' in veh:
            c = 'van'
        if '_cargoBike_' in veh or '_cargobike_' in veh:
            c = 'bike'
        
        # tourlen = 0
        # firstservicedist = None
        # # Loop über alle Tourlinks des Fahrzeuges
        # for link in vehicle_tour[veh]:
        #     # falls erster Service erreicht, Strecke bis dahin speichern
        #     if link == ser[0] and firstservicedist is None:
        #         firstservicedist = tourlen
        #     # Streckenlänge aufsummieren
        #     tourlen += link_length[link]
        # # Sonderfälle abfangen, wo ein Service kurz vor Tagesende beginnt und am nächsten Tag weitergefahren wird
        # if firstservicedist is None: firstservicedist = tourlen

        tourlen = 0
        firstservicedist = None
        service_dists = dict()

        visited_links = vehicle_tour[veh]
        if not ser:
            continue

        for link in visited_links:
            # Strecke aufsummieren
            link_km = link_length[link] / 1000  # Direkt km
            tourlen += link_km
            for s in ser:
                # Sobald Servicepunkt erreicht (einmalig erfassen)
                if link == s and s not in service_dists:
                    service_dists[s] = tourlen
                    if firstservicedist is None:
                        firstservicedist = tourlen  # Speichere ersten Treffer

        # Fallback: falls ein Service nicht in der Tour auftaucht
        for s in ser:
            if s not in service_dists:
                service_dists[s] = tourlen
        if firstservicedist is None:
            firstservicedist = tourlen
           
        indexService = 0
        start_service_distance_count = False
        currentServiceDist = 0
        serviceDistList= []    

        # Service Verteilung über Raumtypen ermitteln
        raumtypen_services = defaultdict(int)
        for s in ser:
            rt = link_raumtyp.get(s, 0) # keinem Raumtyp zugeordnet -> 0
            raumtypen_services[rt] += 1        
    

        summ_service_distances = 0
        previous_time = 0     


        vehicle_stats.append([
            veh, c, len(ser),
            tourlen ,
            firstservicedist ,
            raumtypen_services,
            service_dists,
            firstservicedist   # explizite Spalte für initial_delivery_distance
        ])


    # Dataframe aus Liste
    veh_df = pd.DataFrame(vehicle_stats, columns=[
        'vehicle_id', 'veh_class', 'service_num',
        'tour_km', 'first_service_dist',
        'raumsplit', 'service_dists',
        'initial_delivery_distance'
    ])

    return veh_df

# Definition of help-methods needed for the convertion process

# Method returning the index of an element of a dictionary
def get_nth_key(dictionary, n=0):
    """
    This function returns the nth key of a dictionary.
    
    Args:
        dictionary (dict): The dictionary to get the key from.
        n (int): The index of the key to get. Default is 0.
        
    Returns:
        key: The nth key of the dictionary.
    """
        
    if n < 0:
        n += len(dictionary)
    for i, key in enumerate(dictionary.keys()):
        if i == n:
            return key
    raise IndexError("dictionary index out of range") 

    
# Get Logistic Provider from XML ID
def getProviderFromID(carrierID):
    """
    This function returns the provider name based on the carrier ID.
    
    Args:
        carrierID (str): The ID of the carrier.
        
    Returns:
        str: The name of the provider.
    """
        
    if("dhl" in str(carrierID)):
        return "dhl"  
    elif("amazon" in str(carrierID)):
        return "amazon"  
    elif("ups" in str(carrierID)):
        return "ups"  
    elif("gls" in str(carrierID)):
        return "gls"  
    elif("dpd" in str(carrierID)):
        return "dpd"  
    elif("fedex" in str(carrierID)):
        return "fedex"  
    elif("hermes" in str(carrierID)):
        return "hermes"  
    elif("wl" in str(carrierID)):
        return "White-Label"  
    else:
        raise ValueError('Carrier Provider unknown: ' + str(carrierID))
        
# Definition of needen Classes: Carrier, Vehicle, Plan & Service with Variables

# A carrier object has an ID and dictionaries with all Vehicles and all Services
class Carrier:  
    """
    This class represents a Carrier with an ID and dictionaries with all Vehicles and all Services.
    """
        
    def __init__(self, carrierID): 
        self.carrierID = carrierID 
        self.vehicles = {} 
        self.services = {}   
        self.missedDeliveries = []
        self.logisticProvider = getProviderFromID(self.carrierID)
        
    def __str__(self):
        return "[Carrier ID: " + str(self.carrierID) + " with " + str(len(self.vehicles)) + " Vehicles and " + str(len(self.services)) + " Services]"
    
    def __repr__(self):
        return self.__str__()
    
    def getNumberOfVehicles(self):
        return (len(self.vehicles))
    
    def getNumberOfServices(self):
        return (len(self.services))
        
    def addVehicle(self, vehicle): 
        self.vehicles[vehicle.getVehicleId] = vehicle
        
    def addService(self, service): 
        self.services[service.getServiceID] = service

    def getTotalNumberofServices(self):  
        return sum((s.getDemand() for s in self.services.values()))
    
    def getProvider(self):
        return self.logisticProvider
    
    def getServices(self): 
        return [s for s in self.services.values()]   
    
    def getCarrierId(self): 
        return self.carrierID    
    
    def getVehicles(self): 
        return self.vehicles
    
    def getMissedDeliveries(self): 
        return self.missedDeliveries  
    
    def setMissedDeliveries(self, missDeliveries): 
        self.missedDeliveries = missDeliveries


# A Vehicle has an ID, a Typ and you can add Plans to this vehicles with services and Routes. 
# These Vehicles / Plans have to be converted to Sumo! 
class Vehicle: 
    """
    This class represents a Vehicle with an ID, a Type and you can add Plans to this vehicles with services and Routes.
    """
        
    def __init__(self, vehicleID, vehicleType): 
        self.vehicleID = vehicleID 
        self.vehicleType = vehicleType 
        self.plans = []
        
    def __str__(self):
        return "[Vehicle ID: " + str(self.vehicleID) + " with Type: " + self.vehicleType +"]"
    
    def __repr__(self):
        return self.__str__()
    
    def addVehiclePlan(self, plan): 
        self.plans.append(plan)
        
    def changeVehicleType(self, vehicleType): 
        self.vehicleType == vehicleType
        
    def getVehicleId(self): 
        return self.vehicleID
    
    def getPlans(self):
        return self.plans


# A Plan is a sequence of Activies (start, service, service, ... , end)
# all important Information are stored in the activities / legs dictionary
# The Order of the sequence is stored as a list called "planSequence"
class Plan:  
    """
    This class represents a Plan which is a sequence of Activities (start, service, service, ... , end).
    """
        
    def __init__(self, planId, vehicle, activities, legs): 
        self.planId = planId
        self.vehicle = vehicle
        self.activities = activities 
        self.legs = legs 
        self.planSequence = [] 
        
    def __str__(self):
        return "[Plan ID: " + str(self.planId) + " with " + str(len(self.activities)) +" Activities and " + str(len(self.legs)) + " Legs]"
    
    def __repr__(self):
        return self.__str__()
    
    def getPlanSequence(self):
        return self.planSequence
    
    def createInternalPlanSequence(self):
        for i in range (len(self.legs)):            
            self.planSequence.append(get_nth_key(self.activities, i))
            self.planSequence.append(get_nth_key(self.legs, i))
        self.planSequence.append(get_nth_key(self.activities, len(self.activities)-1))    

# A Service is a data Container storing all availble Infomation from the CSV-File
class Service(): 
    """
    This class represents a Service which is a data Container storing all available Information from the CSV-File.
    """
        
    def __init__(self, serviceType , serviceID, capacityDemand, duration, link, extra_attributes=None):
        self.serviceType = serviceType
        self.serviceID = serviceID 
        self.capacityDemand = capacityDemand 
        self.duration = duration 
        self.link = link 
        self.extra_attributes = extra_attributes or {}
        
    def __str__(self):
        return "[Service ID: " + str(self.serviceID) + "(Type: " + str(self.serviceType) + ") with a CapacityDemand of " + str(self.capacityDemand) +" to Link: " + str(self.link) + "]"
    
    def __repr__(self):
        return self.__str__()

    def getAttribute(self, key, default=None):
        return self.extra_attributes.get(key, default)
    
    def getServiceType(self): 
        return self.serviceType
        
    def changeVehicleType(self, vehicleType): 
        self.vehicleType = vehicleType
        
    def getDemand(self): 
        return self.capacityDemand
        
    def getServiceID(self): 
        return self.serviceID
    
    def getServiceLink(self): 
        return self.link
    
def extract_provider(vehicle_id):
    """
    This function extracts the provider from the vehicle ID.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The name of the provider.
    """
        
    return vehicle_id.split("_")[1]

def extract_veh_size(vehicle_id):
    """
    Extracts the vehicle size from the vehicle ID by looking for 'size_' and returning the next token.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The vehicle size (e.g., 'l', 'm', 'l'), or 'unknown' if not found.
    """
    try:
        parts = vehicle_id.split("size_")
        if len(parts) > 1:
            return parts[1].split("_")[0]
        else:
            return "unknown"
    except Exception as e:
        print(f"Error extracting vehicle size from '{vehicle_id}': {e}")
        return "unknown"

def extract_main_area_type(raumsplit):
    """
    This function extracts the main area type from the raumsplit.
    
    Args:
        raumsplit (dict): The dictionary containing the raumsplit.
        
    Returns:
        str: The main area type.
    """
        
    return max(raumsplit.items(), key=lambda x: x[1])[0]

def process_vehicle_data(veh_df_van):
    """
    This function processes the vehicle data and adds provider, vehicle size, and main area type to the DataFrame.
    
    Args:
        veh_df_van (DataFrame): The DataFrame containing the vehicle data.
        
    Returns:
        DataFrame: The processed DataFrame.
    """
        
    veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
    veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
    veh_df_van['main_area_type'] = veh_df_van['raumsplit'].apply(extract_main_area_type)

    return veh_df_van

def get_vehicles(vehicle_tour):
    """
    This function gets the vehicles from the vehicle tour.
    
    Args:
        vehicle_tour (dict): The dictionary containing the vehicle tour.
        
    Returns:
        list: The list of vehicles.
    """
        
    return [vehicle for vehicle in vehicle_tour if "_supply_" not in vehicle]

def create_plot_data(vehicles, event_file):
    """
    Produces the same outputs as the original function:
    - Computes Start time as earliest 'start' actend per person
    - Computes End time as latest 'end' actstart per person
    - Tour Duration = End - Start
    - Service Duration = sum over paired 'service' actend..actstart intervals per person, in order
    - Travel Duration = Tour Duration - Service Duration
    - Keeps the same columns and the same later string roundtrip before adding formatted columns
    - Excludes persons containing '_supply_'
    """

    # Initialize exactly the same columns with zeros, indexed by vehicles
    plot_data = pd.DataFrame(
        {
            "Start time": 0.0,
            "End time": 0.0,
            "Tour Duration": 0.0,
            "Service Duration": 0.0,
            "Travel Duration": 0.0,
        },
        index=pd.Index(vehicles, name=None),
        dtype="float64",
    )

    # Collect events exactly like before
    events = matsim.event_reader(event_file, types="actstart,actend")

    start_rows = []     # actType == start and type == actend
    end_rows =         []  # actType == end and type == actstart
    service_s_rows = [] # actType == service and type == actend
    service_e_rows = [] # actType == service and type == actstart

    for ev in events:
        person = ev.get("person")
        if not person or "_supply_" in person:
            continue
        etype = ev.get("type")
        atype = ev.get("actType")
        t = ev.get("time", 0.0)

        if atype == "start" and etype == "actend":
            start_rows.append((person, t))
        elif atype == "end" and etype == "actstart":
            end_rows.append((person, t))
        elif atype == "service" and etype == "actend":
            service_s_rows.append((person, t))
        elif atype == "service" and etype == "actstart":
            service_e_rows.append((person, t))

    df_start = pd.DataFrame(start_rows, columns=["person", "time"])
    df_end = pd.DataFrame(end_rows, columns=["person", "time"])
    df_srv_s = pd.DataFrame(service_s_rows, columns=["person", "time"])
    df_srv_e = pd.DataFrame(service_e_rows, columns=["person", "time"])

    # Start time per person: min of start rows
    if not df_start.empty:
        s_min = df_start.groupby("person", sort=False)["time"].min()
        common = plot_data.index.intersection(s_min.index)
        plot_data.loc[common, "Start time"] = s_min.loc[common].to_numpy()

    # End time per person: max of end rows
    if not df_end.empty:
        e_max = df_end.groupby("person", sort=False)["time"].max()
        common = plot_data.index.intersection(e_max.index)
        plot_data.loc[common, "End time"] = e_max.loc[common].to_numpy()

    # Tour Duration
    plot_data.loc[:, "Tour Duration"] = plot_data["End time"] - plot_data["Start time"]

    # Service Duration by pairing actend..actstart in order per person
    service_duration = pd.Series(0.0, index=plot_data.index)
    if not df_srv_s.empty and not df_srv_e.empty:
        g_s = df_srv_s.sort_values(["person", "time"]).groupby("person", sort=False)["time"].apply(list)
        g_e = df_srv_e.sort_values(["person", "time"]).groupby("person", sort=False)["time"].apply(list)
        persons = g_s.index.intersection(g_e.index)
        for pid in persons:
            ls = g_s[pid]
            le = g_e[pid]
            n = min(len(ls), len(le))
            total = 0.0
            for i in range(n):
                total += (le[i] - ls[i])
            if pid in service_duration.index:
                service_duration.loc[pid] = abs(total)

    plot_data.loc[:, "Service Duration"] = service_duration.to_numpy()
    plot_data.loc[:, "Travel Duration"] = plot_data["Tour Duration"] - plot_data["Service Duration"]

    # Keep the original string cast, then reset index and rename like before
    plot_data = plot_data.astype(str)

    plot_data = plot_data.reset_index()
    plot_data = plot_data.rename(columns={"index": "vehicle_id"})

    # Recreate the same formatted time columns from the string values
    plot_data["Start time formatted"] = pd.to_datetime(plot_data["Start time"], unit="s", errors="coerce")
    plot_data["End time formatted"] = pd.to_datetime(plot_data["End time"], unit="s", errors="coerce")
    plot_data["Tour Duration formatted"] = pd.to_datetime(plot_data["Tour Duration"], unit="s", errors="coerce")
    plot_data["Service Duration formatted"] = pd.to_datetime(plot_data["Service Duration"], unit="s", errors="coerce")
    plot_data["Travel Duration formatted"] = pd.to_datetime(plot_data["Travel Duration"], unit="s", errors="coerce")

    # Convert numeric columns back to numeric as in the original tail
    for c in ["Start time", "End time", "Tour Duration", "Service Duration", "Travel Duration"]:
        plot_data[c] = pd.to_numeric(plot_data[c], errors="coerce").fillna(0.0)

    return plot_data

def parse_carriers_from_xml(root):
    """
    This function parses carriers from an XML root.
    
    Args:
        root (ElementTree): The XML root to parse carriers from.
        
    Returns:
        list: The list of carriers.
    """

    carriers = []
    totalVeh = 0
    ns = {"m": "http://www.matsim.org/files/dtd"}

    for carrierXML in root.findall("m:carrier", ns):
        carrierID = carrierXML.attrib.get("id")
        if "supply" in carrierID:
            continue

        newCarrier = Carrier(carrierID)

        for attributeXML in carrierXML.findall("m:attributes", ns):
            for attribute in attributeXML.findall("m:attribute", ns):
                if attribute.attrib.get("name") == "missedParcelDeliveriesAsString":
                    missedParcels = attribute.text
                    missedParcels = missedParcels.replace("[", "").replace("]", "").replace(" ", "")
                    missedServicePerCarrier = missedParcels.split(",")
                    newCarrier.setMissedDeliveries(missedServicePerCarrier)

        for serviceXML in carrierXML.findall(".//m:service", ns):
            serviceID = serviceXML.attrib.get("id")

            serviceCapacityDemand = int(serviceXML.attrib.get("capacityDemand"))
            serviceDuration = serviceXML.attrib.get("serviceDuration")
            serviceLink = serviceXML.attrib.get("to")

            known_keys = {"id", "capacityDemand", "serviceDuration", "to"}
            extra_attrs = {k: v for k, v in serviceXML.attrib.items() if k not in known_keys}

            attributes_element = serviceXML.find("m:attributes", ns)
            if attributes_element is not None:
                for attr in attributes_element.findall("m:attribute", ns):
                    key = attr.attrib.get("name")
                    value = attr.text
                    if key in ["b2b", "b2c"]:
                        try:
                            value = int(value)
                        except ValueError:
                            value = 0
                    extra_attrs[key] = value

            newService = Service(
                "service",
                serviceID,
                serviceCapacityDemand,
                serviceDuration,
                serviceLink,
                extra_attributes=extra_attrs
            )
            newCarrier.addService(newService)

        for plan in carrierXML.findall(".//m:plan", ns):
            if plan.attrib.get("selected") == "true":
                i = 0
                for tour in plan.findall(".//m:tour", ns):
                    i += 1
                    vehID = tour.attrib.get("vehicleId") + "_" + str(i)
                    vehicle = Vehicle(vehID, "cep")                    

                    b = 0
                    c = 0

                    activities = {}
                    legs = {}
                    routes = {}

                    for act in tour.findall(".//m:act", ns):
                        actType = act.attrib.get("type")
                        if actType == "start":
                            activities[actType] = act
                        elif actType == "end":
                            activities[actType] = act
                        else:
                            actID = act.attrib.get("serviceId")
                            activities[actID] = act

                    for leg in tour.findall(".//m:leg", ns):
                        legs["leg_" + str(b)] = leg
                        b += 1

                    for route in tour.findall(".//m:route", ns):
                        if route.text is None:
                            routes["route_" + str(c)] = None
                        else:
                            routes["route_" + str(c)] = route.text.split(" ")
                        c += 1

                    for key, value in legs.items():
                        routeKey = "route_" + str(key.split("_")[1])
                        route = routes.get(routeKey)
                        value.attrib["route"] = route

                    vehiclePlan = Plan("plan_" + vehID, vehicle, activities, legs)
                    vehiclePlan.createInternalPlanSequence()
                    vehicle.addVehiclePlan(vehiclePlan)
                    newCarrier.addVehicle(vehicle)

                totalVeh += newCarrier.getNumberOfVehicles()

        carriers.append(newCarrier)

    return carriers



def calculate_costs(row):
    veh_time_cost = 22.87 / 3600  # Kosten pro Sekunde

    if "size_l" in row['vehicle_id']:
        veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
    elif "size_m" in row['vehicle_id']: 
        veh_cap, veh_fix, veh_km_cost = 165, 171.78, 0.372
    else:
        if "supply_light_van" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
        elif "light" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 1000, 550.63, 0.48643
        else:
            veh_cap, veh_fix, veh_km_cost = 2000, 618.55, 0.555126     
      
     
    vehicle_fix_cost = veh_fix
    vehicle_km_cost = row["tour_km"] * veh_km_cost
    vehicle_time_cost = row["Tour Duration"] * veh_time_cost

    overtime_seconds = max(0, row["Tour Duration"] - (7.5 * 3600))
    overtime_cost = overtime_seconds * veh_time_cost

    vehicle_cost = vehicle_fix_cost + vehicle_km_cost + overtime_cost

    return pd.Series([vehicle_fix_cost, vehicle_km_cost, vehicle_time_cost, overtime_cost, vehicle_cost])



def add_vehicle_demand_to_result(carriers, result):
    
    """
    This function adds vehicle demand to the result DataFrame.
    
    Args:
        carriers (list):The list of carriers.
        result (DataFrame): The DataFrame to add vehicle demand to.
        
    Returns:
        DataFrame: The DataFrame with added vehicle demand.
    """

    required_cols = [
        'deliveries', 'missed deliveries', 'b2b_ration', 'b2c_ration',
        'ration_check', 'vehicle_load_factor', 'vehicle_deliver_factor',
        'vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost',
        'overtime_cost', 'vehicle_cost', 'service_num'
    ]
    for col in required_cols:
        if col not in result.columns:
            result[col] = np.nan
    expected_ids = []
    
        
    veh = 0
    fail = 0

    # 1. Create a new dictionary to hold vehicle-service mappings
    vehicle_services_dict = {}

    for c in tqdm(carriers, desc="Processing carriers", unit="carrier"):
        missedDeliveries = c.getMissedDeliveries()
        missed_deliveries_set = set(missedDeliveries) 

        services = c.getServices()        
        
        for k, v in c.getVehicles().items():
            veh = veh + 1 
            result_df_id = "freight_" + c.getCarrierId()+"_veh_"+v.getVehicleId()     
            expected_ids.append(result_df_id)   
         
            veh_df_res = result[result.vehicle_id == result_df_id]    
            if (len(veh_df_res) != 1):
                print(result_df_id)
                fail = fail + 1    
                
            mask = result.vehicle_id == result_df_id
            if mask.sum() == 0:
                print(f"⚠️ ID not found in result: {result_df_id}")   
            
            totalVehDemand = 0            
            missedParcels = 0

            b2b = 0
            b2c = 0

            # 2. Extract services for this vehicle
            veh_services = []
            services_dict = {s.getServiceID(): s for s in services}

            for a in v.getPlans()[0].activities:
                if "service" in a:
                    for s in services:
                        if a == s.getServiceID():
                            service = services_dict.get(a)
                            veh_services.append(service)

                            service_id = s.getServiceID()
                            service_b2b = int(s.getAttribute("b2b", 0))
                            service_b2c = int(s.getAttribute("b2c", 0))

                            # Nachfrage aufsummieren
                            b2b += service_b2b
                            b2c += service_b2c
                            serviceDemand = service_b2b + service_b2c
                            totalVehDemand += serviceDemand


                            merged = s.getAttribute("mergedMetadata", None)
                            
                            if service_id in missed_deliveries_set:
                                missedParcels += serviceDemand  
                            elif merged:
                                try:
                                    merged_dict = json.loads(merged)

                                    # Validierung: mixed muss zu merged passen
                                    if "MIXED" not in service_id.upper():
                                        print(f"⚠️ Warning: mergedMetadata found, but service ID does not indicate MIXED: {service_id}")

                                    for sid, md in merged_dict.items():
                                        if sid in missed_deliveries_set:
                                            # Sub-service demand: prefer capacity, else len(weights), else 1
                                            sub_cap_raw = md.get("capacity", 0)
                                            try:
                                                sub_cap = int(sub_cap_raw)
                                            except Exception:
                                                sub_cap = 0

                                            sub_weights_raw = md.get("weights", [])
                                            # Normalize weights to a list
                                            if isinstance(sub_weights_raw, str):
                                                try:
                                                    parsed = json.loads(sub_weights_raw)
                                                    sub_weights = parsed if isinstance(parsed, list) else [parsed]
                                                except Exception:
                                                    sub_weights = [sub_weights_raw]
                                            elif isinstance(sub_weights_raw, (list, tuple)):
                                                sub_weights = list(sub_weights_raw)
                                            else:
                                                # float/None/other
                                                sub_weights = []

                                            # Assert consistency if both present
                                            if sub_cap > 0 and len(sub_weights) > 0:
                                                assert sub_cap == len(sub_weights), (
                                                    f"Mismatch in merged service {sid}: "
                                                    f"capacity={sub_cap}, len(weights)={len(sub_weights)}"
                                                )

                                            # Final sub-demand
                                            sub_demand = sub_cap if sub_cap > 0 else (len(sub_weights) if len(sub_weights) > 0 else 1)

                                            # Count only missed sub-services
                                            if sid in missed_deliveries_set:
                                                missedParcels += sub_demand
                                except Exception as e:
                                    print(f"⚠️ Error while parsing mergedMetadata for {service_id}: {e}")


                                
                                    
            vehicle_services_dict[result_df_id] = veh_services                         
            veh_cap = 230 if "size_l" in result_df_id else 165

            b2b_ration = b2b / totalVehDemand
            b2c_ration = b2c / totalVehDemand
            
            result.loc[result.vehicle_id == result_df_id, 'b2b_ration'] = b2b_ration
            result.loc[result.vehicle_id == result_df_id, 'b2c_ration'] = b2c_ration
            result.loc[result.vehicle_id == result_df_id, 'ration_check'] = b2b_ration + b2c_ration

            result.loc[result.vehicle_id == result_df_id, 'deliveries'] = totalVehDemand
            if result[result.vehicle_id == result_df_id].empty:
                print(f"Assignment failed for: {result_df_id}")
            
            result.loc[result.vehicle_id == result_df_id, 'missed deliveries'] = round(missedParcels,0)
            
             # Anwenden der Funktion auf den DataFrame
            cols = ['vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost', 'overtime_cost', 'vehicle_cost']
            result[cols] = result.apply(calculate_costs, axis=1)
             

            result.loc[result.vehicle_id == result_df_id, 'vehicle_load_factor'] = totalVehDemand / veh_cap
            result.loc[result.vehicle_id == result_df_id, 'vehicle_deliver_factor'] = (totalVehDemand - missedParcels) / totalVehDemand
        
    validate_vehicle_id_assignment(result, expected_ids)
    
    result['deliveries_per_stop'] = result['deliveries'] / result['service_num']
    result['Hannover'] = result['vehicle_id'].apply(lambda x: any(plz in x for plz in plzList))

    result['services'] = result['vehicle_id'].map(vehicle_services_dict)
    
    return result

def validate_vehicle_id_assignment(result, expected_ids):
    """
    Validates whether all expected vehicle IDs are present in the result DataFrame.
    Prints helpful diagnostics for debugging.
    
    Args:
        result (DataFrame): The result DataFrame after add_vehicle_demand_to_result.
        expected_ids (list): List of vehicle_id strings that should exist in result.
    """
    actual_ids = result['vehicle_id'].astype(str).tolist()

    missing_ids = [vid for vid in expected_ids if vid not in actual_ids]
    duplicate_ids = result['vehicle_id'][result['vehicle_id'].duplicated()].unique().tolist()
    empty_ids = result['vehicle_id'].isna().sum()

    print("\n📋 Vehicle ID Validation Report")
    print("──────────────────────────────")
    print(f"🔢 Expected IDs total: {len(expected_ids)}")
    print(f"✅ Found: {len(expected_ids) - len(missing_ids)}")
    print(f"❌ Missing: {len(missing_ids)}")
    print(f"🔁 Duplicates: {len(duplicate_ids)}")
    print(f"⚠️ Empty vehicle_id entries: {empty_ids}")
    
    if missing_ids:
        print("\n❌ Missing IDs (first 10 shown):")
        for mid in missing_ids[:10]:
            print(f"  - {mid}")
    
    if duplicate_ids:
        print("\n🔁 Duplicate vehicle_ids:")
        for did in duplicate_ids:
            print(f"  - {did}")



In [87]:
import os
import pickle
import traceback
import pandas as pd
import numpy as np

BATCH_DIR = r"C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch"
OUTPUT_BASE = os.path.join(BATCH_DIR, "processed")  # write alongside matsim processed

DAYEND = 24*3600

# This is a list of postal codes (it seems) that the script will be working with.
plzList = ["30159", "30161", "30163", "30165", "30167", "30169", "30171", "30173", "30175", "30177", "30179", "30419", "30449"
        ,"30451" ,"30453" ,"30455" ,"30457" ,"30459" ,"30519" ,"30521" ,"30539" ,"30559" ,"30625" ,"30627" ,"30629",
        "30625" , "30627", "30629", "30655", "30657", "30659", "30669", "31303", "31303"]

os.makedirs(OUTPUT_BASE, exist_ok=True)

def _safe_save_obj(obj, path):
    base, ext = os.path.splitext(path)
    pkl_path = base + '.pkl'
    print("Writing Results to: ", pkl_path)
    with open(pkl_path, 'wb') as f:
        pickle.dump(obj, f)
    return pkl_path

class SimpleProgress:
    def __init__(self, total: int):
        import time
        self.total = total
        self.start = time.time()
        self.last_len = 0
    def update(self, i: int, prefix: str = ""):
        import time
        now = time.time()
        elapsed = now - self.start
        done = i
        remaining = max(self.total - done, 0)
        eta = (elapsed / done * remaining) if done else 0
        bar_len = 30
        filled = int(bar_len * done / self.total) if self.total else 0
        bar = "#" * filled + "-" * (bar_len - filled)
        msg = f"{prefix} [{bar}] {done}/{self.total} | elapsed {elapsed:6.1f}s | ETA {eta:6.1f}s"
        print("\r" + msg + " " * max(self.last_len - len(msg), 0), end="")
        self.last_len = len(msg)
        if done == self.total:
            print()

def _discover_runs(batch_dir: str):
    runs = []
    if not os.path.isdir(batch_dir):
        return runs
    for scen_entry in sorted(os.listdir(batch_dir)):
        scen_path = os.path.join(batch_dir, scen_entry)
        if not os.path.isdir(scen_path):
            continue
        scenario = scen_entry.split(" ")[0]
        subs = [os.path.join(scen_path, d) for d in os.listdir(scen_path) if os.path.isdir(os.path.join(scen_path, d))]
        search_roots = subs if subs else [scen_path]
        for root in search_roots:
            try:
                files = os.listdir(root)
            except PermissionError:
                continue
            ev = [f for f in files if f.endswith('output_events.xml') or f.endswith('output_events.xml.gz')]
            ca = [f for f in files if f.endswith('output_carriers.xml') or f.endswith('output_carriers.xml.gz')]
            if not ev or not ca:
                continue
            def prefix_of(fn: str) -> str:
                return fn[:-3] if fn.endswith('.gz') else fn
            ev_pref = {prefix_of(f).replace('.output_events.xml',''): f for f in ev}
            ca_pref = {prefix_of(f).replace('.output_carriers.xml',''): f for f in ca}
            for pref in sorted(set(ev_pref) & set(ca_pref)):
                event_file = os.path.join(root, ev_pref[pref])
                carrier_file = os.path.join(root, ca_pref[pref])
                run_name = os.path.basename(root) if root != scen_path else pref
                out_dir = os.path.join(OUTPUT_BASE, scenario, run_name)
                runs.append({
                    'scenario': scenario,
                    'run_name': run_name,
                    'event_file': event_file,
                    'carrier_file': carrier_file,
                    'out_dir': out_dir,
                })
    return runs

def run_batch():

    runs = _discover_runs(BATCH_DIR)
    if not runs:
        print(f"No runs found in {BATCH_DIR}")
        return
    print(f"Discovered {len(runs)} runs across scenarios: {sorted(set([r['scenario'] for r in runs]))}")

    progress = SimpleProgress(total=len(runs))
    rows = []
    for i, r in enumerate(runs, 1):
        # if i > 1:
        #     continue
        
        progress.update(i - 1, prefix=f"Processing {r['scenario']}/{r['run_name']}")
        os.makedirs(r['out_dir'], exist_ok=True)
        try:
            globals()['OUT_DIR'] = r['out_dir']
            # Persist exports regardless (if computed globals are available)
            try:
                # This is a constant representing the number of seconds in a day.

                city = regionclusters[regionclusters.raumtyp < 7]

                result_dataframes = {}
                result_networks= {}

                event_file = r['event_file']
                carrier_file = r['carrier_file']

                print("Start Reading Event-file from: ", event_file)

                vehicle_tour, service_events, network_volumes, vehicle_services = parse_events(event_file)    
                # Clip the network_volumes GeoDataFrame to the shape of gdf_areas
                clipped_network_volumes = gpd.clip(network_volumes, gdf_areas)

                veh_df = vehicle_stats(vehicle_tour, service_events)
                veh_df_van = veh_df[veh_df.veh_class == "van"]

                veh_df_truck = veh_df[~(veh_df.veh_class == "van")]

                # Call the optimized function to process the vehicle data
                veh_df_van = process_vehicle_data(veh_df_van)

                vehicles = get_vehicles(vehicle_tour)
                plot_data = create_plot_data(vehicles, event_file)

                result = plot_data.merge(veh_df_van, left_on='vehicle_id', right_on='vehicle_id')

                with gzip.open(carrier_file, mode="rt") as f:
                    tree = ET.parse(f)
                    root = tree.getroot()

                carriers = parse_carriers_from_xml(root)
                result = add_vehicle_demand_to_result(carriers, result)

                import json
                import re
                from shapely.geometry import Point
                from collections import defaultdict

                vehicle_coords = defaultdict(list)
                vehicle_weights = defaultdict(list)

                for carrier in carriers:
                    services = carrier.getServices()
                    vehicles = carrier.getVehicles()

                    for v in vehicles.values():
                        vehicle_id = f"freight_{carrier.getCarrierId()}_veh_{v.getVehicleId()}"
                        for a in v.getPlans()[0].activities:
                            if "service" not in a:
                                continue

                            for s in services:
                                if a == s.getServiceID():
                                    # --- Normale Koordinate ---
                                    coord_raw = s.getAttribute("coord", None)
                                    if coord_raw:
                                        match = re.match(r"\(([\d\.]+);([\d\.]+)\)", coord_raw)
                                        if match:
                                            x, y = float(match.group(1)), float(match.group(2))
                                            vehicle_coords[vehicle_id].append(Point(x, y))

                                    # --- Normale Gewichte ---
                                    weights_raw = s.getAttribute("weights", None)
                                    if weights_raw:
                                        for w in re.split(r"[;,\s]+", weights_raw.strip()):
                                            try:
                                                weight = max(float(w), 0.1)
                                                vehicle_weights[vehicle_id].append(weight)
                                            except:
                                                pass

                                    # --- Merged Subservices ---
                                    merged = s.getAttribute("mergedMetadata", None)
                                    if merged:
                                        try:
                                            merged_dict = json.loads(merged)
                                            for sub_id, meta in merged_dict.items():
                                                # Koordinaten
                                                if "coord" in meta and isinstance(meta["coord"], list):
                                                    x, y = meta["coord"]
                                                    vehicle_coords[vehicle_id].append(Point(float(x), float(y)))

                                                # Gewichte
                                                if "weights" in meta:
                                                    for w in meta["weights"]:
                                                        try:
                                                            weight = max(float(w), 0.1)
                                                            vehicle_weights[vehicle_id].append(weight)
                                                        except:
                                                            pass
                                        except Exception as e:
                                            print(f"⚠️ Error parsing mergedMetadata for {s.getServiceID()}: {e}")

                # Koordinaten als Liste speichern (Shapely-Points oder Tupel)
                vehicle_coords_df = pd.DataFrame([
                    {"vehicle_id": vid, "coords_list": points}
                    for vid, points in vehicle_coords.items()
                ])

                # Gewichte als Liste speichern
                vehicle_weights_df = pd.DataFrame([
                    {"vehicle_id": vid, "weights_list": ws}
                    for vid, ws in vehicle_weights.items()
                ])

                # Erst Gewichte mergen
                result = result.merge(vehicle_weights_df, how="left", on="vehicle_id")

                # Dann Koordinaten mergen
                result = result.merge(vehicle_coords_df, how="left", on="vehicle_id")

                from shapely.geometry import MultiPoint
                from shapely.ops import transform
                import pyproj

                # Funktion: Fläche in km² berechnen
                project = pyproj.Transformer.from_crs("epsg:25832", "epsg:3857", always_xy=True).transform  # UTM → Meter
                def calc_area_km2(coords):
                    if not coords or len(coords) < 3:
                        return 0.0
                    try:
                        poly = MultiPoint(coords).convex_hull
                        poly_m = transform(project, poly)
                        return round(poly_m.area / 1e6, 3)  # m² → km²
                    except:
                        return 0.0

                # Neue Spalten berechnen
                result["total_weight"] = result["weights_list"].apply(lambda ws: round(sum(ws), 2) if isinstance(ws, list) else 0.0)
                result["avg_weight_per_parcel"] = result["weights_list"].apply(lambda ws: round(sum(ws)/len(ws), 2) if isinstance(ws, list) and ws else 0.0)
                result["delivery_area_km2"] = result["coords_list"].apply(lambda cs: calc_area_km2(cs))


                def calculate_service_dist_metrics(service_dists: dict, initial_dist: float) -> dict:
                    """
                    Calculates average, median, and maximum distance between services,
                    with and without the initial delivery distance.

                    Args:
                        service_dists (dict): Mapping of service_id → cumulative distance (in km)
                        initial_dist (float): Distance to the first delivery (in km)

                    Returns:
                        dict: Contains average, median, and maximum distance with/without initial delivery distance
                    """
                    if not service_dists:
                        return {
                            'avg_dist_with_init': 0,
                            'avg_dist_wo_init': 0,
                            'median_dist_with_init': 0,
                            'median_dist_wo_init': 0,
                            'max_dist_with_init': 0,
                            'max_dist_wo_init': 0
                        }

                    # Sort service distances by tour distance
                    distances = sorted(service_dists.values())

                    # Distance list including initial delivery distance
                    dist_with_init = [initial_dist] + [distances[i + 1] - distances[i] for i in range(len(distances) - 1)]

                    # Distance list excluding initial delivery distance (only between services)
                    dist_wo_init = [distances[i + 1] - distances[i] for i in range(len(distances) - 1)] if len(distances) > 1 else []

                    return {
                        'avg_dist_with_init': sum(dist_with_init) / len(dist_with_init),
                        'avg_dist_wo_init': sum(dist_wo_init) / len(dist_wo_init) if dist_wo_init else 0,
                        'median_dist_with_init': float(np.median(dist_with_init)) if dist_with_init else 0,
                        'median_dist_wo_init': float(np.median(dist_wo_init)) if dist_wo_init else 0,
                        'max_dist_with_init': max(dist_with_init) if dist_with_init else 0,
                        'max_dist_wo_init': max(dist_wo_init) if dist_wo_init else 0
                    }

                # Apply the function to each row in your DataFrame
                metrics = result.apply(
                    lambda row: calculate_service_dist_metrics(row['service_dists'], row['first_service_dist']), axis=1
                )
                metrics_df = pd.DataFrame(metrics.tolist())  # Flatten list of dicts into columns
                result = pd.concat([result, metrics_df], axis=1)


                def determine_area_type(raumtyp):
                    if raumtyp in [1, 2, 3]:
                        return 'Urban'
                    elif raumtyp in [4, 5, 6]:
                        return 'Suburban'
                    elif raumtyp in [7, 8]:
                        return 'Rural'
                    else:
                        return 'unknown'

                # Umbenennen der Spalten für besser lesbare KPIs
                rename_map = {
                    'tour_km': 'Tour Length (km)',
                    'Tour Duration': 'Tour Duration (hrs)',
                    'Travel Duration': 'Driving Time (hrs)',
                    'vehicle_load_factor': 'Vehicle Utilization (%)',
                    'initial_delivery_distance': 'Initial Delivery Distance (km)',
                    'avg_dist_wo_init': 'Mean Distance Between Stops (km)'
                }
                result = result.rename(columns=rename_map)

                # Konvertiere Sekunden → Stunden (nur wenn noch nicht konvertiert)
                for col in ['Tour Duration (hrs)', 'Driving Time (hrs)']:
                    result[col] = result[col] / 3600

                # Durchschnittsgeschwindigkeit berechnen (km/h)
                result['Average Speed (km/h)'] = result['Tour Length (km)'] / result['Driving Time (hrs)']

                result['main_area_type_agg'] = result['main_area_type'].apply(determine_area_type)
                # Prozentualer Anteil der reinen Fahrzeit an der Gesamttourdauer
                result['Driving Share of Tour (%)'] = (result['Driving Time (hrs)'] / result['Tour Duration (hrs)']) * 100

                def determine_area_type(raumtyp):
                    if raumtyp in [1, 2, 3]:
                        return 'Urban'
                    elif raumtyp in [4, 5, 6]:
                        return 'Suburban'
                    elif raumtyp in [7, 8]:
                        return 'Rural'
                    else:
                        return 'unknown'
                    
                result['area_type_agg'] = result['main_area_type'].apply(determine_area_type)
                result['main_area_description'] = result['main_area_type'].map(area_type_dict)

                result = result[result['Vehicle Utilization (%)'] >= 0.05]
       
                _safe_save_obj(result, os.path.join(OUTPUT_BASE, f"{r['run_name']}_result"))

            except Exception as e:
                print('[WARN] run_emissions_pipeline failed:', e)
                rows.append({**r, 'status': f'error: {e.__class__.__name__}: {e}'})
                with open(os.path.join(r['out_dir'], '_impact_error.txt'), 'w') as f:
                    traceback.print_exc(file=f)
                       

        finally:
            progress.update(i, prefix=f"Processing {r['scenario']}/{r['run_name']}")

    summary = pd.DataFrame(rows)
    try:
        summary.to_parquet(os.path.join(OUTPUT_BASE, 'summary_impact.parquet'), index=False)
    except Exception:
        summary.to_pickle(os.path.join(OUTPUT_BASE, 'summary_impact.pkl'))
    print('Impact batch complete. Summary written to', os.path.join(OUTPUT_BASE, 'summary_impact.*'))

In [88]:
print('Batch runner ready: run_batch')
run_batch()
print('All done.')

Batch runner ready: run_batch
Discovered 24 runs across scenarios: ['basecase', 'batchhigh', 'batchmedium', 'batchmoderate']
Processing basecase/basecase_12052025_iter150_jsprit100 [------------------------------] 0/24 | elapsed    0.0s | ETA    0.0sStart Reading Event-file from:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\basecase Week\basecase_12052025_iter150_jsprit100\basecase_12052025.output_events.xml.gz


C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:470: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:472: SettingWithCopyWarning: 
A value is trying to b


📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1651
✅ Found: 1651
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
Writing Results to:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\basecase_12052025_iter150_jsprit100_result.pkl
Processing basecase/basecase_13052025_iter150_jsprit100 [#-----------------------------] 1/24 | elapsed  306.5s | ETA 7048.8sStart Reading Event-file from:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\basecase Week\basecase_13052025_iter150_jsprit100\basecase_13052025.output_events.xml.gz


C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:470: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:472: SettingWithCopyWarning: 
A value is trying to b


📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1746
✅ Found: 1746
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
Writing Results to:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\basecase_13052025_iter150_jsprit100_result.pkl
Processing basecase/basecase_14052025_iter150_jsprit100 [##----------------------------] 2/24 | elapsed  630.6s | ETA 6936.3sStart Reading Event-file from:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\basecase Week\basecase_14052025_iter150_jsprit100\basecase_14052025.output_events.xml.gz


...


📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 2033
✅ Found: 2033
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
Writing Results to:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\batchmoderate_15052025_iter150_jsprit100_result.pkl
Processing batchmoderate/batchmoderate_16052025_iter150_jsprit100 [###########################---] 22/24 | elapsed 6838.9s | ETA  621.7sStart Reading Event-file from:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\batchmoderate Week\batchmoderate_16052025_iter150_jsprit100\batchmoderate_16052025.output_events.xml.gz


C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:470: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:472: SettingWithCopyWarning: 
A value is trying to b


📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1565
✅ Found: 1565
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
Writing Results to:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\batchmoderate_16052025_iter150_jsprit100_result.pkl
Processing batchmoderate/batchmoderate_17052025_iter150_jsprit100 [############################--] 23/24 | elapsed 7115.8s | ETA  309.4sStart Reading Event-file from:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\batchmoderate Week\batchmoderate_17052025_iter150_jsprit100\batchmoderate_17052025.output_events.xml.gz


C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:470: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_34940\3871681668.py:472: SettingWithCopyWarning: 
A value is trying to b


📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1145
✅ Found: 1145
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
Writing Results to:  C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\batchmoderate_17052025_iter150_jsprit100_result.pkl
Processing batchmoderate/batchmoderate_17052025_iter150_jsprit100 [##############################] 24/24 | elapsed 7290.7s | ETA    0.0s
Impact batch complete. Summary written to C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\summary_impact.*
All done.


In [75]:
result_df = pd.read_pickle(r"C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\processed\basecase_12052025_iter150_jsprit100_result.pkl")
